# Xây dựng Knowledge Base và đánh giá QA retrieval

Audit XIV/XVI/XVII + GIR + TT85 → metadata namespace/chunk_type → BGE-M3 dense/sparse → PostgreSQL/pgvector → graph và kiểm chứng. Hai namespace dùng cùng vector store. Biểu mẫu TT85 chỉ nằm trong danh mục template, không embedding. Logic dùng chung trong rag/; kết quả giữ tại output notebook. Hiệu lực pháp lý là metadata nguồn local, chưa xác minh độc lập trong workflow này.


In [1]:
import os
import sys
from pathlib import Path

candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in candidates if (p / "rag/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag.data import config  # noqa: E402 - project root must be resolved first

cfg = config()
print("Workspace:", ROOT)

Workspace: /Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent


In [2]:
import time

import numpy as np
import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import audit, build_corpus
from rag.db import connect, counts, ingest, migrate
from rag.embedding import Embedder, load_embeddings

## Đối chiếu nguồn và sửa đầu vào tìm kiếm — 27/09/2026
Đối chiếu PDF địa phương và Công báo TT31/WCO HS2022. Sửa lỗi OCR đã kiểm chứng; giữ tên pháp lý tiếng Việt 85.06/85.34. Chỉ các trường của chính bản ghi nguồn được đưa vào embedding/BM25/FTS. Chú giải và bản ghi cha giữ riêng với ID, locator và graph; không nối lời dẫn do pipeline tạo vào văn bản ingest.
Search views của XIV/XVI/XVII được dựng lại từ trường tiếng Việt thuộc từng bản ghi master; bản gốc song ngữ giữ để truy vết. Snapshot và benchmark cũ không áp dụng sau khi thay đổi corpus.


Bổ sung 29/09/2026: đối chiếu ảnh PDF Phần XVII tại các trang có Chương 87, sửa `0 tô` thành `Ô tô` trong văn bản chuẩn hóa/ingest, giữ bản trích xuất thô và dấu vết sửa. Không nối mô tả HS6/Heading cha vào văn bản HS8; quan hệ cha được giữ trong graph.


In [3]:
# Authorized source-only corrections, verified against local PDFs and official publications.
import copy
import hashlib
import json
import re
from pathlib import Path

import jsonschema

from rag.data import (
    NORMALIZED,
    ROOT,
    SOURCES,
    read,
    xiv_vietnamese_view,
    xvi_vietnamese_view,
    xvii_vietnamese_view,
)

OLD_8704 = 'Xecó độngcơdùng đểchởhàng.'
VERIFIED_8704 = 'Xe có động cơ dùng để chở hàng.'
VIETNAM_8704_URL = 'https://congbaocdn.chinhphu.vn/CongBaoCP/VanBan/2022/6/37431/41050-1-2022553-55431-2022-tt-btc.pdf'
WCO_85_URL = 'https://www.wcoomd.org/-/media/wco/public/global/pdf/topics/nomenclature/instruments-and-tools/hs-nomenclature-2022/2022/1685_2022e.pdf?la=en'
verified_on = '2026-09-27'
source_evidence = [
    {'target': '87.04', 'local_pdf': 'data/raw/TT31-Section-XVII.pdf', 'pdf_page': 50,
     'printed_page': '1003', 'official_url': VIETNAM_8704_URL, 'official_pdf_page': 58,
     'decision': 'Whitespace correction only; original extracted raw text retained'},
    {'target': '85.06', 'local_pdf': 'data/raw/TT_31-Section-XVI.pdf', 'pdf_page': 147,
     'printed_page': '904', 'official_url': WCO_85_URL, 'official_pdf_page': 6,
     'decision': 'Keep the verified Vietnamese heading; exclude English from retrieval text'},
    {'target': '85.34 / Chapter 85 Note 8', 'local_pdf': 'data/raw/TT_31-Section-XVI.pdf',
     'pdf_page': '179 / 122', 'printed_page': '936 / 886', 'official_url': WCO_85_URL,
     'official_pdf_page': '14 / 2',
     'decision': 'Keep legal heading and Note 8 as separate source records with their own locators'},
]
for evidence in source_evidence:
    evidence['local_pdf_sha256'] = hashlib.sha256((ROOT / evidence['local_pdf']).read_bytes()).hexdigest()
paths = {k: NORMALIZED / SOURCES[k] for k in (
    'tt31_section_xiv_master', 'tt31_section_xiv',
    'tt31_section_xvi_master', 'tt31_section_xvi',
    'tt31_section_xvii_master', 'tt31_section_xvii')}
data = {k: read(p) for k, p in paths.items()}
original = copy.deepcopy(data)
masters = {r['record_id']: r for r in data['tt31_section_xvi_master']}
assert masters['SXVI-T-8506']['text_vi'] == 'Pin và bộ pin.'
assert masters['SXVI-T-8506']['text_en'] == 'Primary cells and primary batteries.'
assert masters['SXVI-T-8534']['text_vi'] == 'Mạch in.'
assert masters['SXVI-T-8534']['text_en'] == 'Printed circuits.'
note = masters['SXVI-N-CH85_NOTE_8']
assert note['text_vi'].startswith('Theo mục đích của nhóm 85.34')
assert 'phần tử kết nối' in note['text_vi'] and '85.42' in note['text_vi']
for sid in ('SXVI-T-8506', 'SXVI-T-8534'):
    m = masters[sid]
    m['retrieval']['source_verification'] = {'verified_on': verified_on, 'official_url': WCO_85_URL,
        'basis': 'Vietnamese statutory heading, with Chapter 85 Note 8 retained as a separate record'}
    m.pop('context_source_ids', None)
    m['retrieval'].pop('source_context', None)

# Correct five bilingual OCR spills using the Vietnamese columns of TT31 Section XVI.
# Page numbers below are PDF pages (printed pages are PDF page + 757).
verified_xvi_texts = {
    'SXVI-T-85362099': ('Loại khác', 181, 'Of a kind used in radio equipment'),
    'SXVI-T-85369039': ('Loại khác', 184, 'Battery clamps, of a kind'),
    'SXVI-T-8538': ('Bộ phận chỉ dùng hoặc chủ yếu dùng với các thiết bị thuộc nhóm 85.35, 85.36 hoặc 85.37.',
                    185, 'with the apparatus of heading'),
    'SXVI-T-85389012': ('Bộ phận của hàng hoá thuộc phân nhóm 8536.50.51, 8536.50.59, 8536.69.32 hoặc 8536.69.39',
                       186, 'or 8536.69.39'),
    'SXVI-T-85491310': ('Của pin và bộ pin', 199, 'Of electric accumulators of a kind'),
}
xvi_source_url = 'https://congbao.chinhphu.vn/van-ban/thong-tu-so-31-2022-tt-btc-37431/41047.htm'
for sid, (vi, pdf_page, spill) in verified_xvi_texts.items():
    m = masters[sid]
    assert m['source']['page_start'] == pdf_page
    previous = m['text_vi']
    assert previous == vi or spill in previous, (sid, previous)
    if previous != vi:
        for descendant in data['tt31_section_xvi_master']:
            for field in ('embedding_text_vi', 'lexical_text_vi'):
                value = descendant['retrieval'].get(field)
                if isinstance(value, str):
                    descendant['retrieval'][field] = value.replace(previous, vi)
        for row in data['tt31_section_xvi']:
            for field in ('text', 'sparse_text'):
                row[field] = row[field].replace(previous, vi)
        m['text_vi'] = vi
    m['retrieval']['source_verification'] = {
        'verified_on': verified_on,
        'local_pdf': 'data/raw/TT_31-Section-XVI.pdf',
        'local_pdf_page': pdf_page,
        'official_url': xvi_source_url,
        'basis': 'Vietnamese tariff column, compared visually with the original PDF; raw English column retained',
    }
source_evidence.extend({
    'target': sid, 'local_pdf': 'data/raw/TT_31-Section-XVI.pdf', 'pdf_page': page,
    'printed_page': str(page + 757), 'official_url': xvi_source_url,
    'decision': 'Remove bilingual OCR spill from normalized Vietnamese and all descendant search views',
    'local_pdf_sha256': hashlib.sha256((ROOT / 'data/raw/TT_31-Section-XVI.pdf').read_bytes()).hexdigest(),
} for sid, (_, page, _) in verified_xvi_texts.items())
display(pd.DataFrame(source_evidence))

for m in data['tt31_section_xvii_master']:
    if m['id'] == 'SXVII_HEADING4_8704':
        assert m['content']['description_vi_normalized'] in (OLD_8704, VERIFIED_8704)
        m['content']['description_vi_normalized'] = VERIFIED_8704
        for key, value in m['retrieval'].items():
            if isinstance(value, str):
                m['retrieval'][key] = value.replace(OLD_8704, VERIFIED_8704).replace(OLD_8704.casefold(), VERIFIED_8704.casefold())
        m['normalization']['verified_whitespace_correction'] = {
            'verified_on': verified_on, 'local_pdf_page': 50, 'official_pdf_page': 58,
            'official_url': VIETNAM_8704_URL, 'raw_extraction_preserved': True}
# Four Chapter 87 samples (PDF pages 13, 49, 56, 62) were checked visually:
# the printed initial is Ô although the PDF text layer and DOCX export read 0.
suspect_oto = re.compile(r"(?<![\w])0(?=\s+tô\b)", re.IGNORECASE)
corrected_oto = []
for m in data['tt31_section_xvii_master']:
    if m['chapter'] != '87':
        continue
    previous = m['content'].get('description_vi_normalized') or ''
    corrected = suspect_oto.sub('Ô', previous)
    if corrected != previous:
        m['content']['description_vi_normalized'] = corrected
        m['normalization']['verified_ocr_correction'] = {
            'pattern': '0 tô -> Ô tô', 'verified_on': '2026-09-29',
            'basis': 'Printed Vietnamese tariff column; four page samples visually reviewed',
            'sample_pdf_pages': [13, 49, 56, 62], 'raw_extraction_preserved': True}
        corrected_oto.append(m['id'])
total_oto = sum(m.get('normalization', {}).get('verified_ocr_correction', {}).get('pattern') == '0 tô -> Ô tô'
                for m in data['tt31_section_xvii_master'])
source_evidence.append({'target': 'Section XVII Chapter 87',
    'local_pdf': 'data/raw/TT31-Section-XVII.pdf', 'pdf_page': '13 / 49 / 56 / 62',
    'decision': f'Normalize OCR initial 0 to printed Ô in {total_oto} Chapter 87 descriptions ({len(corrected_oto)} changed this run); retain raw extraction',
    'local_pdf_sha256': hashlib.sha256((ROOT / 'data/raw/TT31-Section-XVII.pdf').read_bytes()).hexdigest()})
print('Chapter 87 OCR search-view corrections:', total_oto, 'total;', len(corrected_oto), 'changed this run')

xvi_masters = {m['record_id']: m for m in data['tt31_section_xvi_master']}
for m in data['tt31_section_xvi_master']:
    own_text = xvi_vietnamese_view(m)
    assert own_text
    m['retrieval']['embedding_text_vi'] = own_text
    m['retrieval']['lexical_text_vi'] = own_text
for row in data['tt31_section_xvi']:
    own_text = xvi_vietnamese_view(xvi_masters[row['id']])
    row['text'] = row['sparse_text'] = own_text
    row['metadata'].pop('context_source_ids', None)

xiv_masters = {m['id']: m for m in data['tt31_section_xiv_master']}
for row in data['tt31_section_xiv']:
    m = xiv_masters[row['source_record_id']]
    vi_text = xiv_vietnamese_view(m)
    row['content']['embedding_text'] = vi_text
    row['content']['sparse_text'] = vi_text
    row['content']['search_text'] = vi_text
    row['content']['fts']['text_en'] = None
    row['content']['exact_keys'] = [v for v in (
        m['hs'].get('code'), m['hs'].get('code_compact'), m['text'].get('vi')) if v]
    for ancestor in row['hierarchy']['ancestor_descriptions']:
        ancestor['en'] = None
    row['search_language'] = 'vi'
assert all(r['search_language'] == 'vi' for r in data['tt31_section_xiv'])

xvii_masters = {m['id']: m for m in data['tt31_section_xvii_master']}
for row in data['tt31_section_xvii']:
    m = xvii_masters[row['source_node_id']]
    vi_text = xvii_vietnamese_view(m)
    row['content']['embedding_text'] = vi_text
    row['content']['fts_text'] = vi_text
    row['language_profile'] = 'vi'
    m['retrieval']['search_text_vi'] = vi_text
    m['retrieval']['search_text'] = vi_text
    m['retrieval']['search_text_casefold'] = vi_text.casefold()
assert all(r['language_profile'] == 'vi' and '\nEN:' not in r['content']['embedding_text']
    and not suspect_oto.search(r['content']['embedding_text'])
    for r in data['tt31_section_xvii'])

schema = read(NORMALIZED / SOURCES['tt31_section_xvi_schema'])
for m in data['tt31_section_xvi_master']:
    jsonschema.Draft202012Validator(schema).validate(m)
for key in paths:
    old_rows, new_rows = original[key], data[key]
    id_field = ('record_id' if key == 'tt31_section_xvi_master' else
        'source_record_id' if key == 'tt31_section_xiv' else
        'source_node_id' if key == 'tt31_section_xvii' else 'id')
    assert [r[id_field] for r in old_rows] == [r[id_field] for r in new_rows]
    for old, new in zip(old_rows, new_rows, strict=True):
        if key == 'tt31_section_xvi_master':
            assert old['text_en'] == new['text_en'] and old['source'] == new['source']
            if old['record_id'] in verified_xvi_texts:
                assert new['text_vi'] == verified_xvi_texts[old['record_id']][0]
            else:
                assert old['text_vi'] == new['text_vi']
        elif key == 'tt31_section_xvii_master':
            assert old['content']['description_vi_raw'] == new['content']['description_vi_raw']
            assert old['code'] == new['code'] and old['parent_id'] == new['parent_id']
        elif key == 'tt31_section_xvii':
            assert old['content']['raw_authoritative'] == new['content']['raw_authoritative']
            assert old['metadata'] == new['metadata'] and old['graph'] == new['graph']
master_lookup = {r['record_id']: r for r in data['tt31_section_xvi_master']}
assert all(r['text'] == master_lookup[r['id']]['retrieval']['embedding_text_vi'] for r in data['tt31_section_xvi'])

# Keep untouched JSONL lines byte-for-byte; no additional data/report files.
repair_rows = []
for key, path in paths.items():
    lines = path.read_text().splitlines(keepends=True)
    assert len(lines) == len(data[key])
    changed = 0
    for i, (before, after) in enumerate(zip(original[key], data[key], strict=True)):
        if before != after:
            lines[i] = json.dumps(after, ensure_ascii=False) + '\n'
            changed += 1
    if changed:
        path.write_text(''.join(lines))
    repair_rows.append({'file': str(path.relative_to(ROOT)), 'changed_rows_this_run': changed,
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
display(pd.DataFrame(repair_rows))
print('PASS: IDs/codes, authoritative heading wording, raw extraction and graph relationships preserved.')
print('Sections XIV/XVI/XVII ingest text: each record uses only its own Vietnamese source fields; raw source retained for audit.')
print('Source-only correction complete. This cell does not activate a database snapshot or change retrieval policies.')
print('The subsequent build/embedding/ingestion cells apply these inputs when that workflow is run; prior benchmark outputs remain historical.')


,target,local_pdf,pdf_page,printed_page,official_url,official_pdf_page,decision,local_pdf_sha256
0,87.04,data/raw/TT31-Section-XVII.pdf,50,1003,https://congbaocdn.chinhphu.vn/CongBaoCP/VanBa...,58,Whitespace correction only; original extracted...,00a24df2696660b5f6ad06f37d01f28d2e64b8d5743992...
1,85.06,data/raw/TT_31-Section-XVI.pdf,147,904,https://www.wcoomd.org/-/media/wco/public/glob...,6,Keep the verified Vietnamese heading; exclude ...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
2,85.34 / Chapter 85 Note 8,data/raw/TT_31-Section-XVI.pdf,179 / 122,936 / 886,https://www.wcoomd.org/-/media/wco/public/glob...,14 / 2,Keep legal heading and Note 8 as separate sour...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
3,SXVI-T-85362099,data/raw/TT_31-Section-XVI.pdf,181,938,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
4,SXVI-T-85369039,data/raw/TT_31-Section-XVI.pdf,184,941,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
5,SXVI-T-8538,data/raw/TT_31-Section-XVI.pdf,185,942,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
6,SXVI-T-85389012,data/raw/TT_31-Section-XVI.pdf,186,943,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...
7,SXVI-T-85491310,data/raw/TT_31-Section-XVI.pdf,199,956,https://congbao.chinhphu.vn/van-ban/thong-tu-s...,NaN,Remove bilingual OCR spill from normalized Vie...,812b348badb5d6b57748a56551d0d1bec580cc74b2a732...


Chapter 87 OCR search-view corrections: 173 total; 0 changed this run


,file,changed_rows_this_run,sha256
0,data/normalized/TT31/Section-XIV/SECTION_XIV_M...,0,9b3104316a405e6a9c65a575cfa23165ad179975d14318...
1,data/normalized/TT31/Section-XIV/SECTION_XIV_R...,0,dc1ab212db56e19adc30544a6d7afece76e7ace40cd130...
2,data/normalized/TT31/Section-XVI/SECTION_XVI_M...,0,8ffd185f646d089f3b373a0516c47514e785f17095d4ab...
3,data/normalized/TT31/Section-XVI/SECTION_XVI_R...,0,0ad83d3e188cc0a5131d7406383754b6d8e859087d6d58...
4,data/normalized/TT31/Section-XVII/SECTION_XVII...,0,ce7299ecf342afcc7f23304e24a69e4d4fba12b23ebf61...
5,data/normalized/TT31/Section-XVII/SECTION_XVII...,0,0791389cbc72ccf69a56429432780a95c6a61c68508bd5...


PASS: IDs/codes, authoritative heading wording, raw extraction and graph relationships preserved.
Sections XIV/XVI/XVII ingest text: each record uses only its own Vietnamese source fields; raw source retained for audit.
Source-only correction complete. This cell does not activate a database snapshot or change retrieval policies.
The subsequent build/embedding/ingestion cells apply these inputs when that workflow is run; prior benchmark outputs remain historical.


## 1. Audit và truy vết nguồn
TT31 ingest được nối với master; GIR combined và TT85 source là allowlist. data/verified/hs/train, test và holdout được dùng kiểm coverage mã, không đưa vào corpus.

In [4]:
from rag.data import NORMALIZED, SOURCES

audit_result = audit()
display(pd.DataFrame(audit_result["quality"]).T[["count", "duplicate_wording"]])
display(
    pd.DataFrame(
        [
            {
                "scope_chapters": audit_result["scope_count"],
                "graph_edges": audit_result["graph_edges"],
                "broken_endpoints": len(audit_result["broken_endpoints"]),
                "gold_codes_missing": len(audit_result["gold_missing"]),
                "TT85_SHA256_match": audit_result["tt85_source_hash_matches"],
                "legal_verification": audit_result["legal_verification"],
            }
        ]
    )
)



# Runtime allowlist stays at its original paths. Historical audit files are nested.
layout = []
for section in ("XIV", "XVI", "XVII"):
    folder = NORMALIZED / "TT31" / ("Section-" + section)
    runtime_files = sorted(Path(value).name for value in SOURCES.values()
        if str(Path(value).parent) == "TT31/Section-" + section)
    supporting_files = sorted(p.name for p in (folder / "supporting").iterdir() if p.is_file())
    assert all((folder / name).is_file() for name in runtime_files)
    layout.append({"section": section, "runtime_inputs": runtime_files,
        "supporting_count": len(supporting_files), "supporting": supporting_files})
display(pd.DataFrame(layout))
print("Auxiliary exports are excluded from the ingestion allowlist.")


,count,duplicate_wording
tt31_section_xvi,3421,173
gir,115,0
tt85,100,0
tt31_section_xiv,190.0,3.0
tt31_section_xvii,1433.0,4.0


,scope_chapters,graph_edges,broken_endpoints,gold_codes_missing,TT85_SHA256_match,legal_verification
0,97,5813,0,0,True,LOCAL_METADATA_ONLY


,section,runtime_inputs,supporting_count,supporting
0,XIV,"[SECTION_XIV_GRAPH_EDGES.jsonl, SECTION_XIV_GR...",6,"[SECTION_XIV_CORRECTION_LOG.json, SECTION_XIV_..."
1,XVI,"[SECTION_XVI_GRAPH_EDGES.jsonl, SECTION_XVI_GR...",4,"[SECTION_XVI_EXACT_CODE_INDEX.csv, SECTION_XVI..."
2,XVII,"[SECTION_XVII_GRAPH_EDGES_NORMALIZED.jsonl, SE...",12,[SECTION_XVII_BENCHMARK_FINAL_CORRECTIONS.json...


Auxiliary exports are excluded from the ingestion allowlist.


## 2. Corpus và metadata thống nhất
GIR giữ source atomic và context view; TT85 giữ điều khoản, loại 5 forms khỏi RAG. Structural/context/footnote chunks giữ nguyên vai trò và chunk_type=null khi không thuộc 7 loại statutory/tariff đã chốt, tránh đổi tên footnote thành điều luật. Scope index ngoài 3 Sections chỉ hỗ trợ nhận biết phạm vi.


In [5]:
bundle = build_corpus()
records = pd.DataFrame(bundle["records"])
display(
    records.groupby(["namespace", "dataset", "section", "role"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)
display(
    records.groupby(["chunk_type", "level"], dropna=False).size().rename("records").reset_index()
)
print("Snapshot:", bundle["snapshot"], "Corpus:", len(records))
assert len(records) == 5356
assert records["id"].is_unique
assert not records["id"].str.contains("TT85-FORM-").any()
assert set(records["namespace"]) == {"hs_classification", "legal_classification"}

# Confirm corrected heading text is inside the rebuilt corpus, not just on disk.
corrected_headings = [r for r in bundle["records"] if r["section"] == "XVII"
    and r["level"] == 4 and r["code"] in ("8711", "8716")]
assert len(corrected_headings) == 2
display(pd.DataFrame([{ "id": r["id"], "code": r["code"],
    "source_text": r["source_text"], "embedding_text": r["embedding_text"],
    "input_hash": r["input_hash"]} for r in corrected_headings]))
print("Current input hashes:", {key: value for key, value in bundle["manifest"]["source_hashes"].items()
    if "Section-XVII" in key})


,namespace,dataset,section,role,records
0,hs_classification,gir,NaN,evidence,115
1,hs_classification,scope,I,scope,5
2,hs_classification,scope,II,scope,9
3,hs_classification,scope,III,scope,1
4,hs_classification,scope,IV,scope,9
5,hs_classification,scope,IX,scope,3
6,hs_classification,scope,V,scope,3
7,hs_classification,scope,VI,scope,11
8,hs_classification,scope,VII,scope,2
9,hs_classification,scope,VIII,scope,3


,chunk_type,level,records
0,chapter_note,NaN,102
1,gir_explanation,NaN,104
2,gir_rule,NaN,11
3,legal_clause,NaN,100
4,section_note,NaN,34
5,subheading_note,NaN,15
6,tariff_entry,2.0,5
7,tariff_entry,4.0,191
8,tariff_entry,6.0,645
9,tariff_entry,8.0,3480


Snapshot: e2354dac4a5bdc2c717cac99 Corpus: 5356


,id,code,source_text,embedding_text,input_hash
0,tt31:SXVII_HEADING4_8711,8711,Mô tô (kể cả xe gắn máy có bàn đạp (moped)) và...,Mã 87.11. Mô tô (kể cả xe gắn máy có bàn đạp (...,8c69c82729c0ec6479a8ff228df3e178c5515a9963339a...
1,tt31:SXVII_HEADING4_8716,8716,"Rơ-moóc và sơ mi rơ-moóc; xe khác, không có cơ...","Mã 87.16. Rơ-moóc và sơ mi rơ-moóc; xe khác, k...",ccd3d9b44781435654bba6a2643267b2c202bd5f5cffa0...


Current input hashes: {'TT31/Section-XVII/SECTION_XVII_RAG_INGEST.jsonl': '0791389cbc72ccf69a56429432780a95c6a61c68508bd553e14bbfb2c20ac3df', 'TT31/Section-XVII/SECTION_XVII_MASTER_NORMALIZED.jsonl': 'ce7299ecf342afcc7f23304e24a69e4d4fba12b23ebf612a8732b682c2c1f04f', 'TT31/Section-XVII/SECTION_XVII_GRAPH_EDGES_NORMALIZED.jsonl': '5662495d6bbd4237cf82adc3044dc8120b3e14513fc4cbae9f19112dc92989e4'}


In [6]:
from rag.data import NORMALIZED, read

examples = [
    r
    for r in bundle["records"]
    if r["id"] in ["gir:GIR_3_NOTE_038", "gir:GIR_5_NOTE_016", "tt85:TT85-A9-3.a.1"]
]
display(
    pd.DataFrame(
        [
            {
                "id": r["id"],
                "source_text": r["source_text"],
                "embedding_view": r["embedding_text"],
                "locator": r["locator"],
            }
            for r in examples
        ]
    )
)

forms = read(NORMALIZED / "TT85/tt85_forms_source.json")
display(
    pd.DataFrame(
        [
            {
                "form": f["form_no"],
                "code": f["code"],
                "title": f["title"],
                "fields": len(f["fields"]),
            }
            for f in forms
        ]
    )
)
print("5 forms are a template registry, not RAG chunks; Word generation is not implemented.")
assert len(forms) == 5

,id,source_text,embedding_view,locator
0,gir:GIR_3_NOTE_038,Phân loại vào nhóm 16.02.,GIR Quy tắc 3 | GIR_EXPLANATORY | (a) Bộ thực...,"{'document': 'Phụ lục II - 6_GIR.pdf', 'pages'..."
1,gir:GIR_5_NOTE_016,(Bao bì),GIR Quy tắc 5 | GIR_EXPLANATORY | (Bao bì)\nQ...,"{'document': 'Phụ lục II - 6_GIR.pdf', 'pages'..."
2,tt85:TT85-A9-3.a.1,a.1) Cơ quan hải quan thực hiện tiếp nhận và k...,"85/2026/TT-BTC | Điều 9: Hồ sơ, thủ tục đối vớ...","{'document': '85/2026/TT-BTC', 'article': 9, '..."


,form,code,title,fields
0,01,01/PYCPT/2026,"Phiếu yêu cầu phân tích hàng hóa xuất khẩu, nh...",14
1,02,02/PTNYCPT/2026,Phiếu tiếp nhận yêu cầu phân tích hàng hóa xuấ...,9
2,03,03/BBTLMHH/2026,Biên bản trả lại mẫu đã tiếp nhận yêu cầu phân...,11
3,04,04/TBKQPTPL/2026,"Thông báo về kết quả phân tích, phân loại đối ...",15
4,05,05/PGTL/2026,"Phiếu ghi số, ngày văn bản, chứng từ thuộc hồ ...",6


5 forms are a template registry, not RAG chunks; Word generation is not implemented.


## 3. BGE-M3 dense + sparse
Pin model/tokenizer commit. Corpus input không được truncate im lặng. Checkpoint theo batch/input hash giúp tái chạy nhanh. CPU FP32 và sparse là cấu hình đã chọn cho experiment; dense là 1.024 chiều, sparse là token weights của model.

In [7]:
embedder = Embedder(cfg)
started = time.perf_counter()
with capture_output() as embedding_progress:
    embedding_manifest = embedder.corpus(bundle)
manifest, vectors, sparse = load_embeddings(bundle)
display(
    pd.DataFrame(
        [
            {
                "records": len(vectors),
                "dimension": vectors.shape[1],
                "max_input_tokens": manifest["max_tokens"],
                "sparse_nonzero": sum(len(w) for w in sparse),
                "elapsed_seconds": round(time.perf_counter() - started, 2),
                "model_revision": manifest["profile"]["revision"],
                "profile": manifest["profile_id"],
            }
        ]
    )
)
np.testing.assert_allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)
assert len(sparse) == len(vectors)
assert np.isfinite(vectors).all()

/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Fetching 22 files: 100%|██████████| 22/22 [00:00<00:00, 4016.13it/s]

,records,dimension,max_input_tokens,sparse_nonzero,elapsed_seconds,model_revision,profile
0,5356,1024,523,96226,0.78,5617a9f61b028005a4858fdac845db406aefb181,3a2ce6e6d726a0f053748717


## 4. PostgreSQL: xóa snapshot cũ và nạp lại từ nguồn đã kiểm chứng

Chế độ `FRESH_REBUILD=True` xóa **chỉ** các bảng trong schema `rag` ở database đang hiển thị. Xóa và nạp snapshot mới nằm trong cùng một transaction: nếu ingestion hoặc kiểm chứng lỗi, PostgreSQL khôi phục dữ liệu cũ. Các nguồn `data/raw`, `data/normalized`, `data/verified` và output notebook không bị xóa. BGE-M3 có thể tái sử dụng cache embedding theo hash của văn bản không đổi; database vẫn được nạp lại từ trạng thái rỗng.


In [8]:
conn = connect()
migrate(conn)
display(pd.DataFrame([conn.execute("SELECT current_database() AS database, version() AS server").fetchone()]))
APPLY = True
FRESH_REBUILD = True
RAG_TABLES = (
    "retrieval_results", "evaluation_runs", "active", "sparse_postings",
    "embeddings", "graph_edges", "graph_nodes", "chunks", "snapshots", "profiles",
)
actual_tables = {row["table_name"] for row in conn.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema='rag' AND table_type='BASE TABLE'"
).fetchall()}
assert actual_tables == set(RAG_TABLES), (actual_tables, RAG_TABLES)
before_rebuild = {table: conn.execute(f"SELECT count(*) AS n FROM rag.{table}").fetchone()["n"]
                  for table in RAG_TABLES}
print("RAG database before fresh rebuild:", before_rebuild)
print("Apply:", APPLY, "fresh rebuild:", FRESH_REBUILD)


,database,server
0,hs_rag,PostgreSQL 17.6 (Debian 17.6-1.pgdg12+1) on aa...


RAG database before fresh rebuild: {'retrieval_results': 0, 'evaluation_runs': 0, 'active': 1, 'sparse_postings': 96226, 'embeddings': 5356, 'graph_edges': 6655, 'graph_nodes': 5521, 'chunks': 5356, 'snapshots': 1, 'profiles': 1}
Apply: True fresh rebuild: True


In [9]:
if APPLY and FRESH_REBUILD:
    # All statements run on the isolated RAG schema; no CASCADE into other schemas.
    with conn.transaction():
        conn.execute("SELECT pg_advisory_xact_lock(20260926)")
        conn.execute("TRUNCATE TABLE " + ", ".join(f"rag.{table}" for table in RAG_TABLES))
        assert all(conn.execute(f"SELECT count(*) AS n FROM rag.{table}").fetchone()["n"] == 0
                   for table in RAG_TABLES)
        print("Old RAG tables cleared inside transaction.")
        ingestion_result = ingest(conn, bundle, manifest, vectors, sparse, apply=True)
        assert conn.execute("SELECT count(*) AS n FROM rag.snapshots").fetchone()["n"] == 1
        assert conn.execute("SELECT count(*) AS n FROM rag.chunks").fetchone()["n"] == len(bundle["records"])
        assert conn.execute("SELECT snapshot FROM rag.active").fetchone()["snapshot"] == bundle["snapshot"]
else:
    ingestion_result = ingest(conn, bundle, manifest, vectors, sparse, apply=APPLY)
display(pd.DataFrame([ingestion_result.get("counts", ingestion_result.get("expected"))]))
if APPLY:
    after = counts(conn, bundle["snapshot"], manifest["profile_id"])
    assert after == ingestion_result["counts"]
    print("Fresh snapshot and counts: PASS")
    display(pd.DataFrame([conn.execute("SELECT * FROM rag.active").fetchone()]))


Old RAG tables cleared inside transaction.


,chunks,embeddings,nodes,edges
0,5356,5356,5521,6655


Fresh snapshot and counts: PASS


,singleton,snapshot,profile
0,True,e2354dac4a5bdc2c717cac99,3a2ce6e6d726a0f053748717


## 5. Kiểm chứng cuối
Counts, FK integrity, vector norms và sparse coverage đều được kiểm trực tiếp. Phần QA phía dưới và notebook 02 dùng snapshot/profile active này; không cần giữ kernel hiện tại.

In [10]:
if APPLY:
    check = conn.execute(
        """SELECT count(*) AS sparse_postings FROM rag.sparse_postings
        WHERE snapshot=%s AND profile=%s""",
        (bundle["snapshot"], manifest["profile_id"]),
    ).fetchone()
    assert check["sparse_postings"] == sum(len(w) for w in sparse)
    print(
        "Knowledge base ready:", {"snapshot": bundle["snapshot"], "records": len(vectors), **check}
    )
conn.close()

Knowledge base ready: {'snapshot': 'e2354dac4a5bdc2c717cac99', 'records': 5356, 'sparse_postings': 96226}


## Definition graph audit for retrieval
The active knowledge base already contains Notes and definition edges. Derive cited search views from that active snapshot; do not re-ingest gold or change atomic source wording. Search views are built in memory for BM25 and optional evidence-aware reranking. Existing BGE dense/sparse vectors remain valid. Notebook 02 benchmarks and freezes this retrieval policy.

In [11]:
import pandas as pd
from IPython.display import display

from rag.db import connect
from rag.retrieval import Retriever

with connect() as audit_connection:
    definition_audit = Retriever(audit_connection)
    definition_views = definition_audit.definition_views()
    active_before = audit_connection.execute("SELECT * FROM rag.active").fetchone()
    display(pd.DataFrame([{"section": view["record"]["section"],
        "heading": view["record"]["heading"], "ahtn8": view["record"]["code"],
        "source_ids": [x["id"] for x in view["citations"]],
        "graph_paths": view["paths"]} for view in definition_views.values()]))
    assert all(view["record"]["source_text"] == definition_audit.exact(view["record"]["code"])[0]["source_text"]
        for view in definition_views.values())
    assert audit_connection.execute("SELECT * FROM rag.active").fetchone() == active_before
    print("Source-backed definition views:", len(definition_views), "Active snapshot:", active_before)


,section,heading,ahtn8,source_ids,graph_paths
0,XIV,7113,71132090,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
1,XIV,7113,71132010,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
2,XIV,7113,71131990,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
3,XIV,7113,71131910,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
4,XIV,7113,71131190,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
5,XIV,7113,71131110,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relatio..."
6,XIV,7114,71142000,[tt31:XIV_CH71_NOTE_10],"[{'note_id': 'tt31:XIV_CH71_NOTE_10', 'relatio..."
7,XIV,7114,71141900,[tt31:XIV_CH71_NOTE_10],"[{'note_id': 'tt31:XIV_CH71_NOTE_10', 'relatio..."
8,XIV,7114,71141100,[tt31:XIV_CH71_NOTE_10],"[{'note_id': 'tt31:XIV_CH71_NOTE_10', 'relatio..."
9,XIV,7117,71179099,"[tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_09]","[{'note_id': 'tt31:XIV_CH71_NOTE_11', 'relatio..."


Source-backed definition views: 49 Active snapshot: {'singleton': True, 'snapshot': 'e2354dac4a5bdc2c717cac99', 'profile': '3a2ce6e6d726a0f053748717'}


## QA retrieval trên cùng Knowledge Base

Phần này được gộp từ notebook 03. Nó đo truy xuất căn cứ cho GIR, Chú giải, dòng biểu thuế và TT85 trên **cùng snapshot**; riêng TT85 thuộc namespace `legal_classification`. Chỉ tập QA train chọn trọng số/chính sách; QA test dùng để đánh giá. Kết quả và từng câu nằm ngay trong output bên dưới. Chính sách HS được giữ khi cập nhật khóa `qa`.


In [12]:
import time

import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import config, digest, load_corpus, read, write_json
from rag.db import connect
from rag.embedding import Embedder
from rag.evaluation import (
    CONFIGURATIONS,
    QA_CHUNK_TYPES,
    branch_cache,
    discover_qa,
    qa_direct_coverage,
    qa_objective,
    sweep_qa_weights,
)
from rag.retrieval import Retriever

pd.set_option("display.max_colwidth", 140)

bundle = load_corpus()
conn = connect()
with capture_output():
    embedder = Embedder(config())
runtime = Retriever(conn, embedder)
assert runtime.snapshot == bundle["snapshot"]
print("Active snapshot:", runtime.snapshot, "profile:", runtime.profile)
print("Corpus records:", len(bundle["records"]))

Active snapshot: e2354dac4a5bdc2c717cac99 profile: 3a2ce6e6d726a0f053748717
Corpus records: 5356


## 1. Kiểm tra train và nguồn

Chỉ đọc `data/verified/qa/train/LEGAL_RETRIEVAL_TRAIN.jsonl` ở bước chọn cấu hình. Mapping gold được dùng để **chấm điểm**, không đi vào query hay cơ sở tri thức.

In [13]:
qa_train = discover_qa(bundle, split="train")
assert len(qa_train) == 400
assert len({c["case_family_id"] for c in qa_train}) == len(qa_train)
assert all(c["canonical_groups"] for c in qa_train)
train_hash = digest(qa_train)
source_by_id = {r["id"]: r for r in bundle["records"]}
train_rows = []
for case in qa_train:
    primary = [g["any_of"][0] for g in case["canonical_groups"]]
    train_rows.append(
        {
            "case_type": case.get("case_type", ""),
            "section_scope": ",".join(case.get("section_scope", [])),
            "primary_tariff": any(source_by_id[s]["chunk_type"] == "tariff_entry" for s in primary),
            "primary_count": len(primary),
        }
    )
print("Train:", len(qa_train), "unique queries:", len({c["query"].casefold() for c in qa_train}))
display(
    pd.DataFrame(train_rows)
    .groupby(["case_type", "primary_tariff"])
    .size()
    .rename("count")
    .reset_index()
)
print("Train hash:", train_hash)

Train: 400 unique queries: 400


,case_type,primary_tariff,count
0,cross_section,False,1
1,cross_section,True,6
2,gir,False,37
3,legal_rule,False,9
4,note_exclusion,False,12
5,note_rule,False,94
6,tariff_direct,True,135
7,tariff_plus_note,True,8
8,tt85,False,98


Train hash: 6abe2fd43c2fbf45db3905440b7db7e033c041a3062a3c5913bba3180a74a827


## 2. Chọn retriever và trọng số trên train

Thử các họ Dense/Sparse/BM25/FTS/Exact và Weighted RRF như notebook 02. Chọn theo tỷ lệ **đủ mọi căn cứ primary trong Top-10**, rồi Recall và MRR. Đây là chọn retrieval policy, không fine-tune BGE-M3.

In [14]:
with capture_output():
    train_cache = branch_cache(
        runtime,
        qa_train,
        intent="knowledge",
        pool=100,
        filters={"chunk_types": QA_CHUNK_TYPES},
        mask_questioned_codes=True,
    )
family_best = []
for family in CONFIGURATIONS:
    family_best.append(sweep_qa_weights(train_cache, family)[0])
base_choice = max(family_best, key=qa_objective)
base_policy = {
    "configuration": base_choice["configuration"],
    "weights": base_choice["weights"],
    "k": base_choice["k"],
    "pool": 100,
    "reranker": "cross_encoder",
    "rerank_pool": 30,
    "rank_weight": 0.75,
    "graph_before_rerank": False,
    "query_split_weight": 0.0,
}
old_policy = {
    "configuration": "C11",
    "weights": {"dense": 0.30, "sparse": 0.05, "bm25": 0.30, "fts": 0.05, "exact": 0.30},
    "k": 5,
    "pool": 100,
    "reranker": "cross_encoder",
    "rerank_pool": 30,
    "rank_weight": 0.75,
    "graph_before_rerank": False,
    "query_split_weight": 0.0,
}
display(
    pd.DataFrame(family_best)[
        [
            "configuration",
            "weights",
            "k",
            "all_groups@5",
            "all_groups@10",
            "group_recall@10",
            "mrr@10",
        ]
    ]
)
print("Train-selected base:", base_policy)
print("Previous snapshot baseline is evaluated as a transfer, not assumed optimal.")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


,configuration,weights,k,all_groups@5,all_groups@10,group_recall@10,mrr@10
0,C1,{'dense': 1.0},5,0.8100,0.8375,0.851750,0.716423
1,C2,{'sparse': 1.0},5,0.7825,0.8475,0.854667,0.696544
2,C3,{'bm25': 1.0},5,0.7100,0.7700,0.785500,0.642606
3,C4,{'fts': 1.0},5,0.2675,0.3775,0.398542,0.170622
4,C5,"{'dense': 0.5056255836134853, 'sparse': 0.4943744163865147}",20,0.8300,0.8900,0.895083,0.742296
5,C6,"{'dense': 0.6006673387057762, 'bm25': 0.39933266129422385}",20,0.8225,0.8850,0.893833,0.720612
6,C7,"{'dense': 0.45675863649883747, 'sparse': 0.44711229497230237, 'fts': 0.09612906852886034}",20,0.8275,0.8900,0.895083,0.737730
7,C8,"{'dense': 0.32176139576680746, 'sparse': 0.3146757434488862, 'exact': 0.36356286078430633}",20,0.8350,0.8975,0.901333,0.730098
8,C9,"{'dense': 0.4060767915011878, 'bm25': 0.2709279840057974, 'exact': 0.32299522449301465}",20,0.8300,0.8950,0.901333,0.719706
9,C10,"{'dense': 0.4, 'sparse': 0.1, 'bm25': 0.2, 'fts': 0.05, 'exact': 0.25}",60,0.8325,0.9025,0.911333,0.718169


Train-selected base: {'configuration': 'C10', 'weights': {'dense': 0.4, 'sparse': 0.1, 'bm25': 0.2, 'fts': 0.05, 'exact': 0.25}, 'k': 60, 'pool': 100, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.75, 'graph_before_rerank': False, 'query_split_weight': 0.0}
Previous snapshot baseline is evaluated as a transfer, not assumed optimal.


## 3. Ablation trên 400 train

Giữ cùng trọng số đã chọn và thay vị trí graph, kích thước pool, tỷ lệ giữ ứng viên trực tiếp, mức pha thứ hạng gốc và truy vấn bỏ mã nghi vấn. Cả heading/Notes mới đều phải là chunks có văn bản nguồn và locator. Latency bên dưới đo trên mô hình đã nạp, cache reranker có thể làm lần sau nhanh hơn; không dùng latency này để chọn chất lượng.

In [15]:
variants = {
    "baseline_old_snapshot": old_policy,
    "direct_30": base_policy,
    "direct_50": {**base_policy, "rerank_pool": 50},
    "graph_30_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 30,
        "direct_fraction": 0.70,
    },
    "graph_50_50": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.50,
    },
    "graph_50_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
    },
    "graph_50_85": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.85,
    },
    "graph_100_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 100,
        "direct_fraction": 0.70,
    },
    "graph_50_pure": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "rank_weight": 1.0,
    },
    "graph_50_neutral_25": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "query_split_weight": 0.25,
    },
    "graph_50_neutral_50": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "query_split_weight": 0.50,
    },
}

with capture_output():
    runtime.qa_search(qa_train[0]["query"], policy_override=base_policy, include_package=False)
train_experiments = []
for name, policy in variants.items():
    started = time.perf_counter()
    for case in qa_train:
        with capture_output():
            result = runtime.qa_search(case["query"], policy_override=policy, include_package=False)
        metric = qa_direct_coverage(case, result["matches"])
        required = {g["any_of"][0] for g in case["canonical_groups"]}
        pool_ids = set(result["pre_rerank_ids"])
        train_experiments.append(
            {
                "variant": name,
                "case": case["id"],
                "case_type": case.get("case_type", ""),
                "section_scope": ",".join(case.get("section_scope", [])),
                "pool_complete": required <= pool_ids,
                "pool_recall": len(required & pool_ids) / len(required),
                "all_groups@5": metric["all_groups@5"],
                "all_groups@10": metric["all_groups@10"],
                "group_recall@10": metric["group_recall@10"],
                "mrr@10": metric["mrr@10"],
                "elapsed_ms": result["elapsed_ms"],
                "query_split_used": result["query_split_used"],
            }
        )
    print(name, "seconds:", round(time.perf_counter() - started, 1), flush=True)
train_frame = pd.DataFrame(train_experiments)
train_summary = (
    train_frame.groupby("variant")
    .agg(
        count=("case", "count"),
        pool_complete=("pool_complete", "mean"),
        pool_recall=("pool_recall", "mean"),
        **{
            "all_groups@5": ("all_groups@5", "mean"),
            "all_groups@10": ("all_groups@10", "mean"),
            "group_recall@10": ("group_recall@10", "mean"),
            "mrr@10": ("mrr@10", "mean"),
        },
        median_ms=("elapsed_ms", "median"),
        split_queries=("query_split_used", "sum"),
    )
    .reset_index()
)
display(train_summary.sort_values(["all_groups@10", "group_recall@10", "mrr@10"], ascending=False))
display(
    train_frame.groupby(["variant", "case_type"])["all_groups@10"]
    .agg(["mean", "count"])
    .reset_index()
)


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


baseline_old_snapshot seconds: 108.2


direct_30 seconds: 61.4


direct_50 seconds: 82.2


graph_30_70 seconds: 66.9


...

graph_50_pure seconds: 42.3


graph_50_neutral_25 seconds: 42.1


graph_50_neutral_50 seconds: 42.3


,variant,count,pool_complete,pool_recall,all_groups@5,all_groups@10,group_recall@10,mrr@10,median_ms,split_queries
0,baseline_old_snapshot,400,0.9300,0.935917,0.8675,0.9050,0.910083,0.778332,79.678208,0
5,graph_50_50,400,0.9400,0.941750,0.8675,0.9050,0.907583,0.787096,80.433063,0
6,graph_50_70,400,0.9400,0.941750,0.8675,0.9050,0.907583,0.787062,79.638521,0
7,graph_50_85,400,0.9425,0.944250,0.8675,0.9050,0.907583,0.787062,76.762771,0
1,direct_30,400,0.9375,0.940917,0.8675,0.9050,0.907583,0.785721,76.495021,0
2,direct_50,400,0.9425,0.944250,0.8675,0.9050,0.907583,0.785062,86.226563,0
3,graph_100_70,400,0.9450,0.948417,0.8675,0.9050,0.907583,0.784766,522.827646,0
8,graph_50_neutral_25,400,0.9400,0.941750,0.8675,0.9050,0.907583,0.784562,77.723770,5
10,graph_50_pure,400,0.9400,0.941750,0.8600,0.9050,0.907583,0.775899,80.243021,0
4,graph_30_70,400,0.9325,0.937167,0.8675,0.9025,0.906333,0.784634,84.289104,0


,variant,case_type,mean,count
0,baseline_old_snapshot,cross_section,0.571429,7
1,baseline_old_snapshot,gir,0.972973,37
2,baseline_old_snapshot,legal_rule,0.444444,9
3,baseline_old_snapshot,note_exclusion,1.000000,12
4,baseline_old_snapshot,note_rule,0.978723,94
...,...,...,...,...
83,graph_50_pure,note_exclusion,1.000000,12
84,graph_50_pure,note_rule,0.989362,94
85,graph_50_pure,tariff_direct,0.814815,135
86,graph_50_pure,tariff_plus_note,1.000000,8


## 4. Đóng băng QA policy trước khi mở test

Lấy phương án tốt nhất trên train trong phạm vi đã thử. Nếu graph không thắng, nó vẫn được báo trong bảng test để chẩn đoán, nhưng không được âm thầm đưa vào runtime.

In [16]:
candidate_names = list(variants)
new_candidate_names = [name for name in variants if name != "baseline_old_snapshot"]
quality = {name: train_summary.set_index("variant").loc[name] for name in candidate_names}
chosen_name = max(
    candidate_names,
    key=lambda name: (
        quality[name]["all_groups@10"],
        quality[name]["group_recall@10"],
        quality[name]["all_groups@5"],
        quality[name]["mrr@10"],
    ),
)
selected_policy = {
    **variants[chosen_name],
    "selection_status": "QA_400_TRAIN_ONLY",
    "train_sha256": train_hash,
    "train_count": len(qa_train),
    "tested_variants": len(variants),
    "selected_variant": chosen_name,
    "objective": ["all_groups@10", "group_recall@10", "all_groups@5", "mrr@10"],
    "candidate_types": QA_CHUNK_TYPES,
    "limitations": "Best among tested policies on supplied train; no model fine-tuning.",
}
hs_selection = read(ROOT / "config/selected_retrieval.json")
assert hs_selection["profile"] == runtime.profile
if hs_selection["snapshot"] != runtime.snapshot:
    # Carry the prior HS policy temporarily; notebook 02 retunes it on HS train.
    hs_selection = {**hs_selection, "snapshot": runtime.snapshot,
        "selection_scope": "PROVISIONAL_POLICY_TRANSFER; HS_TRAIN_RETUNE_IN_NOTEBOOK_02"}
assert hs_selection["snapshot"] == runtime.snapshot
selected_config = {**hs_selection, "qa": selected_policy}
assert {k: v for k, v in selected_config.items() if k != "qa"} == {
    k: v for k, v in hs_selection.items() if k != "qa"
}
write_json(ROOT / "config/selected_retrieval.json", selected_config)
frozen_hash = digest(selected_config)
print("Frozen before test:", chosen_name, selected_policy)
print("Configuration digest:", frozen_hash)

Frozen before test: baseline_old_snapshot {'configuration': 'C11', 'weights': {'dense': 0.3, 'sparse': 0.05, 'bm25': 0.3, 'fts': 0.05, 'exact': 0.3}, 'k': 5, 'pool': 100, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.75, 'graph_before_rerank': False, 'query_split_weight': 0.0, 'selection_status': 'QA_400_TRAIN_ONLY', 'train_sha256': '6abe2fd43c2fbf45db3905440b7db7e033c041a3062a3c5913bba3180a74a827', 'train_count': 400, 'tested_variants': 11, 'selected_variant': 'baseline_old_snapshot', 'objective': ['all_groups@10', 'group_recall@10', 'all_groups@5', 'mrr@10'], 'candidate_types': ['gir_rule', 'gir_explanation', 'section_note', 'chapter_note', 'subheading_note', 'tariff_entry', 'tariff_footnote', 'legal_clause'], 'limitations': 'Best among tested policies on supplied train; no model fine-tuning.'}
Configuration digest: 874e876c632e9356b2e93fbcba1c833c3974e13c3beb105d4c6341a4edd70d14


## 5. Test 150 câu sau khi đóng băng

So sánh cách cũ, chính sách tìm trực tiếp, graph tốt nhất trên train, và policy đã chọn. Báo riêng 50 câu cũ đã từng được dùng để chẩn đoán; không dùng test để đổi trọng số hay chọn lại policy.

In [17]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
qa_test = discover_qa(bundle, split="test")
assert len(qa_test) == 150
assert not ({c["case_family_id"] for c in qa_train} & {c["case_family_id"] for c in qa_test})
train_best_direct = max(
    (n for n in new_candidate_names if not variants[n]["graph_before_rerank"]),
    key=lambda n: (
        quality[n]["all_groups@10"],
        quality[n]["group_recall@10"],
        quality[n]["mrr@10"],
    ),
)
train_best_graph = max(
    (n for n in new_candidate_names if variants[n]["graph_before_rerank"]),
    key=lambda n: (
        quality[n]["all_groups@10"],
        quality[n]["group_recall@10"],
        quality[n]["mrr@10"],
    ),
)
test_variants = dict.fromkeys(
    ["baseline_old_snapshot", train_best_direct, train_best_graph, chosen_name]
)
test_rows = []
for name in test_variants:
    policy = variants[name]
    started = time.perf_counter()
    for case in qa_test:
        with capture_output():
            result = runtime.qa_search(case["query"], policy_override=policy, include_package=True)
        metric = qa_direct_coverage(case, result["matches"])
        required = {g["any_of"][0] for g in case["canonical_groups"]}
        pool_ids = set(result["pre_rerank_ids"])
        package_ids = {r["id"] for r in result["package"]}
        test_rows.append(
            {
                "variant": name,
                "case": case["id"],
                "query": case["query"],
                "source_group": "old_50_diagnostic"
                if case.get("legacy_original_index") is not None
                else "new_100",
                "case_type": case.get("case_type", ""),
                "section_scope": ",".join(case.get("section_scope", [])),
                "pool_complete": required <= pool_ids,
                "pool_recall": len(required & pool_ids) / len(required),
                "all_groups@5": metric["all_groups@5"],
                "all_groups@10": metric["all_groups@10"],
                "group_recall@10": metric["group_recall@10"],
                "mrr@10": metric["mrr@10"],
                "package_complete": required <= package_ids,
                "package_recall": len(required & package_ids) / len(required),
                "missing_primary": sorted(required - {r["id"] for r in result["matches"]}),
                "returned_top10": [r["id"] for r in result["matches"]],
                "elapsed_ms": result["elapsed_ms"],
                "query_split_used": result["query_split_used"],
            }
        )
    print("TEST", name, "seconds:", round(time.perf_counter() - started, 1), flush=True)
test_frame = pd.DataFrame(test_rows)
columns = [
    "pool_complete",
    "pool_recall",
    "all_groups@5",
    "all_groups@10",
    "group_recall@10",
    "mrr@10",
    "package_complete",
    "package_recall",
]
test_summary = (
    test_frame.groupby("variant")[columns]
    .mean()
    .join(test_frame.groupby("variant").elapsed_ms.agg(["median", lambda s: s.quantile(0.95)]))
)
test_summary.columns = columns + ["latency_p50_ms", "latency_p95_ms"]
display(test_summary)
display(test_frame.groupby(["source_group", "variant"])[columns].mean())
display(
    test_frame.groupby(["case_type", "variant"])[
        ["all_groups@10", "group_recall@10", "package_complete"]
    ].mean()
)
print(
    "Frozen policy digest unchanged:",
    digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash,
)

TEST baseline_old_snapshot seconds: 21.1


TEST direct_30 seconds: 21.6


TEST graph_50_50 seconds: 60.0


,pool_complete,pool_recall,all_groups@5,all_groups@10,group_recall@10,mrr@10,package_complete,package_recall,latency_p50_ms,latency_p95_ms
variant,,,,,,,,,,
baseline_old_snapshot,0.873333,0.876667,0.80,0.846667,0.861111,0.738127,0.893333,0.896667,107.419167,197.223352
direct_30,0.873333,0.883333,0.82,0.853333,0.867778,0.739079,0.893333,0.900000,114.633250,258.617025
graph_50_50,0.900000,0.906667,0.82,0.853333,0.871111,0.740312,0.886667,0.893333,375.458604,720.714817


pool_complete  pool_recall  \
source_group      variant                                             
new_100           baseline_old_snapshot           0.87        0.870   
                  direct_30                       0.87        0.875   
                  graph_50_50                     0.88        0.885   
old_50_diagnostic baseline_old_snapshot           0.88        0.890   
                  direct_30                       0.88        0.900   
                  graph_50_50                     0.94        0.950   

                                         all_groups@5  all_groups@10  \
source_group      variant                                              
new_100           baseline_old_snapshot          0.83           0.86   
                  direct_30                      0.85           0.86   
                  graph_50_50                    0.85           0.86   
old_50_diagnostic baseline_old_snapshot          0.74           0.82   
                  direct_30                      0.76           0.84   
                  graph_50_50                    0.76           0.84   

                                         group_recall@10    mrr@10  \
source_group      variant                                            
new_100           baseline_old_snapshot         0.865000  0.791429   
                  direct_30                     0.870000  0.798095   
                  graph_50_50                   0.870000  0.798095   
old_50_diagnostic baseline_old_snapshot         0.853333  0.631524   
                  direct_30                     0.863333  0.621048   
                  graph_50_50                   0.873333  0.624746   

                                         package_complete  package_recall  
source_group      variant                                                  
new_100           baseline_old_snapshot              0.88           0.885  
                  direct_30                          0.88           0.890  
                  graph_50_50                        0.87           0.880  
old_50_diagnostic baseline_old_snapshot              0.92           0.920  
                  direct_30                          0.92           0.920  
                  graph_50_50                        0.92           0.920

all_groups@10  group_recall@10  \
case_type        variant                                                 
cross_section    baseline_old_snapshot       0.333333         0.500000   
                 direct_30                   0.000000         0.333333   
                 graph_50_50                 0.000000         0.500000   
gir              baseline_old_snapshot       0.937500         0.979167   
                 direct_30                   0.937500         0.979167   
                 graph_50_50                 0.937500         0.979167   
legal_rule       baseline_old_snapshot       1.000000         1.000000   
                 direct_30                   1.000000         1.000000   
                 graph_50_50                 1.000000         1.000000   
note_exclusion   baseline_old_snapshot       1.000000         1.000000   
                 direct_30                   1.000000         1.000000   
                 graph_50_50                 1.000000         1.000000   
note_rule        baseline_old_snapshot       1.000000         1.000000   
                 direct_30                   1.000000         1.000000   
                 graph_50_50                 1.000000         1.000000   
tariff_direct    baseline_old_snapshot       0.770270         0.770270   
                 direct_30                   0.783784         0.783784   
                 graph_50_50                 0.783784         0.783784   
tariff_plus_note baseline_old_snapshot       0.884615         0.923077   
                 direct_30                   0.923077         0.942308   
                 graph_50_50                 0.923077         0.942308   
tt85             baseline_old_snapshot       1.000000         1.000000   
                 direct_30                   1.000000         1.000000   
                 graph_50_50                 1.000000         1.000000   

                                        package_complete  
case_type        variant                                  
cross_section    baseline_old_snapshot          1.000000  
                 direct_30                      0.666667  
                 graph_50_50                    0.666667  
gir              baseline_old_snapshot          1.000000  
                 direct_30                      1.000000  
                 graph_50_50                    1.000000  
legal_rule       baseline_old_snapshot          1.000000  
                 direct_30                      1.000000  
                 graph_50_50                    1.000000  
note_exclusion   baseline_old_snapshot          1.000000  
                 direct_30                      1.000000  
                 graph_50_50                    1.000000  
note_rule        baseline_old_snapshot          1.000000  
                 direct_30                      1.000000  
                 graph_50_50                    1.000000  
tariff_direct    baseline_old_snapshot          0.810811  
                 direct_30                      0.824324  
                 graph_50_50                    0.810811  
tariff_plus_note baseline_old_snapshot          0.923077  
                 direct_30                      0.923077  
                 graph_50_50                    0.923077  
tt85             baseline_old_snapshot          1.000000  
                 direct_30                      1.000000  
                 graph_50_50                    1.000000

Frozen policy digest unchanged: True


In [18]:
base_test = test_frame[test_frame.variant == train_best_direct].set_index("case")
selected_test = test_frame[test_frame.variant == chosen_name].set_index("case")
comparison = selected_test[
    ["query", "source_group", "case_type", "section_scope", "missing_primary", "returned_top10"]
].copy()
comparison["direct_all@10"] = base_test["all_groups@10"]
comparison["selected_all@10"] = selected_test["all_groups@10"]
comparison["change"] = comparison["selected_all@10"].astype(int) - comparison[
    "direct_all@10"
].astype(int)
print(
    "Gains:",
    (comparison.change > 0).sum(),
    "losses:",
    (comparison.change < 0).sum(),
    "unchanged:",
    (comparison.change == 0).sum(),
)
display(comparison.loc[comparison.change != 0].head(30))
display(
    test_summary[["pool_complete", "all_groups@10", "package_complete"]]
    .style.format("{:.1%}")
    .bar(axis=0, color="#87a9d8", vmin=0, vmax=1)
    .set_caption("QA 150 test: đủ căn cứ trước rerank / Top-10 / gói căn cứ")
)


Gains: 2 losses: 3 unchanged: 145


,query,source_group,case_type,section_scope,missing_primary,returned_top10,direct_all@10,selected_all@10,change
case,,,,,,,,,
LQ012,"Máy in trình bày riêng dù có thể kết nối với máy tính có được phân loại vào 84.71 không, và heading nào cần xem xét?",old_50_diagnostic,tariff_plus_note,XVI,[tt31:SXVI-T-8443],"[tt31:SXVI-N-CH84_NOTE_6D, tt31:SXVI-N-CH84_NOTE_6C, tt31:SXVI-T-8471, tt31:SXVI-T-84669330, tt31:SXVI-T-85284200, tt31:SXVI-T-85286200,...",True,False,-1
LQ016,Ắc quy ion liti có thể nạp lại thuộc phân nhóm nào?,old_50_diagnostic,tariff_direct,XVI,[],"[tt31:SXVI-N-CH85_SUBNOTE_5, tt31:SXVI-T-84831010, tt31:SXVI-T-84735010, tt31:SXVI-N-CH85_NOTE_3, tt31:SXVI-T-850760, tt31:SXVI-T-848340...",False,True,1
LQ017,"Pin sơ cấp, không thể nạp lại, cần phân biệt với ắc quy nạp lại ở heading nào?",old_50_diagnostic,tariff_direct,XVI,[tt31:SXVI-T-8506],"[tt31:SXVI-N-CH85_SUBNOTE_5, tt31:SXVI-T-85044020, tt31:SXVI-T-8513, tt31:SXVI-T-85493100, tt31:SXVI-T-85492100, tt31:SXVI-T-854991, tt3...",True,False,-1
LQ063,Hãy xác định dòng AHTN 8 số dùng làm căn cứ trực tiếp cho “Vòng”.,new_100,tariff_direct,XIV,[tt31:HS8:71171910],"[tt85:TT85-A2-8, tt85:TT85-A6-1.c, tt31:SXVII_HS6_870911, tt31:SXVI-T-85043121, tt31:SXVII_HS6_860110, tt31:SXVI-T-84553000, tt31:SXVI-T...",True,False,-1
LQ125,Một mặt hàng thuộc 84.83 dùng trên phương tiện chỉ được coi là bộ phận Phần XVII trong điều kiện nào? Truy xuất Chú giải 2 Phần XVII và ...,new_100,cross_section,"XVI,XVII",[],"[tt31:SXVII_NOTE_2, tt31:SXVI-N-SXVI_NOTE_2B, tt31:SXVI-N-SXVI_NOTE_2_INTRO, tt31:SXVI-N-SXVI_NOTE_2A, tt31:SXVII_NOTE_3, tt31:SXVI-T-84...",False,True,1


,pool_complete,all_groups@10,package_complete
variant,,,
baseline_old_snapshot,87.3%,84.7%,89.3%
direct_30,87.3%,85.3%,89.3%
graph_50_50,90.0%,85.3%,88.7%


## 6. Runtime và kiểm tra

Đối chiếu runtime dùng cấu hình đã đóng băng với đường thử nghiệm. Kiểm tra chính sách HS không bị thay đổi khi thêm khóa QA; kết quả HS và QA thuộc cùng snapshot.

In [19]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
with connect() as fresh_conn:
    fresh = Retriever(fresh_conn, embedder)
    assert fresh.qa_policy["selected_variant"] == chosen_name
    for case in qa_train[:5] + qa_test[:5]:
        with capture_output():
            expected = runtime.qa_search(
                case["query"], policy_override=selected_policy, include_package=False
            )
            actual = fresh.qa_search(case["query"], include_package=False)
        assert [r["id"] for r in expected["matches"]] == [r["id"] for r in actual["matches"]]
print("QA runtime parity: 10/10 exact Top-10.")
assert {k: v for k, v in read(ROOT / "config/selected_retrieval.json").items() if k != "qa"} == {
    k: v for k, v in hs_selection.items() if k != "qa"
}
print("HS configuration unchanged by QA selection.")

QA runtime parity: 10/10 exact Top-10.
HS configuration unchanged by QA selection.


In [20]:
import subprocess

validation_env = {**os.environ, "RAG_RUN_DB_TESTS": "1", "TOKENIZERS_PARALLELISM": "false"}
for command in [
    [str(ROOT / ".venv/bin/ruff"), "check", "rag", "tests", "notebooks"],
    [str(ROOT / ".venv/bin/mypy"), "rag"],
    [str(ROOT / ".venv/bin/bandit"), "-r", "rag", "-ll", "-q"],
    [str(ROOT / ".venv/bin/python"), "-m", "pytest", "tests", "-q"],
    ["git", "diff", "--check"],
]:
    checked = subprocess.run(
        command, cwd=ROOT, env=validation_env, capture_output=True, text=True, check=False
    )
    print(
        " ".join(command),
        "exit",
        checked.returncode,
        checked.stdout[-700:] or checked.stderr[-700:] or "PASS",
    )
    assert checked.returncode == 0
selection_now = read(ROOT / "config/selected_retrieval.json")
assert selection_now["snapshot"] == runtime.snapshot
if "frozen_hash" in globals():
    assert digest(selection_now) == frozen_hash
print("Completed: snapshot", bundle["snapshot"], "QA policy", selection_now["qa"]["selected_variant"])

/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/ruff check rag tests notebooks exit 0 All checks passed!



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/mypy rag exit 0 Success: no issues found in 6 source files



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/bandit -r rag -ll -q exit 0 PASS


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/python -m pytest tests -q exit 0 ........................................................                 [100%]
56 passed in 22.07s

git diff --check exit 0 PASS
Completed: snapshot e2354dac4a5bdc2c717cac99 QA policy baseline_old_snapshot


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## QA regression on the new active Knowledge Base snapshot

The HS corpus was refreshed from the verified source. Recheck the already selected QA policy on all 150 supplied QA test questions without changing its weights or the frozen HS policy. These are retrieval metrics, not legal conclusions.

In [21]:
import numpy as np

from rag.evaluation import qa_coverage

policy_hash = digest(read(ROOT / "config/selected_retrieval.json"))
runtime = Retriever(conn, embedder)
assert runtime.qa_policy["selected_variant"] == chosen_name
assert runtime.snapshot == read(ROOT / "config/selected_retrieval.json")["snapshot"]
qa_test_current = discover_qa(bundle, split="test")
assert len(qa_test_current) == 150
qa_rows = []
for index, case in enumerate(qa_test_current, 1):
    with capture_output() as progress:
        result = runtime.qa_search(case["query"])
    direct = qa_direct_coverage(case, result["matches"])
    package = qa_coverage(case, result["matches"], result["package"])
    qa_rows.append({"id": case["id"], "category": case["category"],
        "direct_all@5": direct["all_groups@5"],
        "direct_all@10": direct["all_groups@10"],
        "direct_group_recall@10": direct["group_recall@10"],
        "direct_mrr@10": direct["mrr@10"],
        "package_all": package["package_all_groups"],
        "package_group_recall": package["package_group_recall"],
        "missing_package": package["missing_package_groups"],
        "latency_ms": result["elapsed_ms"],
        "top10_ids": direct["returned_top10"]})
    if index % 25 == 0:
        print("QA regression", index, "/", len(qa_test_current), flush=True)
assert digest(read(ROOT / "config/selected_retrieval.json")) == policy_hash
frame = pd.DataFrame(qa_rows)
summary = frame.groupby("category").agg(
    questions=("id", "count"),
    direct_all_at_5=("direct_all@5", "mean"),
    direct_all_at_10=("direct_all@10", "mean"),
    direct_group_recall_at_10=("direct_group_recall@10", "mean"),
    direct_mrr_at_10=("direct_mrr@10", "mean"),
    package_all=("package_all", "mean"),
    latency_p50_ms=("latency_ms", lambda x: float(np.percentile(x, 50))),
    latency_p95_ms=("latency_ms", lambda x: float(np.percentile(x, 95))))
display(summary)
print("Overall direct all@10", frame["direct_all@10"].mean(),
      "package all", frame["package_all"].mean())
print("QA policy hash unchanged", policy_hash)
display(frame)



QA regression 25 / 150


QA regression 50 / 150


QA regression 75 / 150


QA regression 100 / 150


QA regression 125 / 150


QA regression 150 / 150


,questions,direct_all_at_5,direct_all_at_10,direct_group_recall_at_10,direct_mrr_at_10,package_all,latency_p50_ms,latency_p95_ms
category,,,,,,,,
cross_section_exclusion,2,0.000000,0.000000,0.250000,0.500000,1.000000,145.685021,150.135089
gir_rule_retrieval,4,0.750000,0.750000,0.916667,0.812500,1.000000,76.187500,92.301415
heading_or_subheading_legal_retrieval,29,0.827586,0.862069,0.862069,0.631609,0.896552,97.360417,195.517275
heading_plus_legal_note,11,0.636364,0.818182,0.863636,0.455411,0.909091,114.296000,154.912542
legal_evidence,55,0.927273,0.981818,0.990909,0.929870,0.981818,129.110792,202.055084
legal_note_retrieval,2,1.000000,1.000000,1.000000,1.000000,1.000000,132.872521,181.961877
multi_source_legal_retrieval,2,0.500000,1.000000,1.000000,1.000000,1.000000,84.516980,87.897436
tariff_lookup,45,0.711111,0.711111,0.711111,0.622222,0.755556,134.417791,250.697875


Overall direct all@10 0.8466666666666667 package all 0.8933333333333333
QA policy hash unchanged 874e876c632e9356b2e93fbcba1c833c3974e13c3beb105d4c6341a4edd70d14


,id,category,direct_all@5,direct_all@10,direct_group_recall@10,direct_mrr@10,package_all,package_group_recall,missing_package,latency_ms,top10_ids
0,LQ001,heading_or_subheading_legal_retrieval,True,True,1.0,1.00,True,1.0,[],1267.699875,"[tt31:HS4:7101, tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_01, tt31:XIV_CH71_NOTE_09, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7103, tt31:HS4:7..."
1,LQ002,heading_or_subheading_legal_retrieval,True,True,1.0,1.00,True,1.0,[],99.158375,"[tt31:HS4:7102, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7103, tt31:HS4:7101, tt31:XIV_CH71_NOTE_03, tt31:HS4:7104, tt31:HS4:7111, tt31:XIV_CH..."
2,LQ003,heading_or_subheading_legal_retrieval,True,True,1.0,0.20,True,1.0,[],78.836708,"[tt31:HS4:7102, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7101, tt31:XIV_CH71_NOTE_11, tt31:HS4:7103, tt31:HS4:7104, tt31:XIV_CH71_NOTE_03, tt3..."
3,LQ004,heading_or_subheading_legal_retrieval,True,True,1.0,0.25,True,1.0,[],80.318833,"[tt31:SXVI-N-SXVI_NOTE_1F, tt31:XIV_CH71_NOTE_11, tt31:HS4:7102, tt31:HS4:7104, tt31:XIV_CH71_NOTE_03, tt31:HS4:7101, tt31:HS4:7103, tt3..."
4,LQ005,heading_or_subheading_legal_retrieval,True,True,1.0,1.00,True,1.0,[],77.010583,"[tt31:HS4:7108, tt31:HS4:7111, tt31:HS4:7106, tt31:HS4:7110, tt31:HS4:7102, tt31:XIV_CH71_NOTE_10, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS8:71..."
...,...,...,...,...,...,...,...,...,...,...,...
145,LQ146,legal_evidence,True,True,1.0,1.00,True,1.0,[],83.405334,"[tt85:TT85-A12-3, tt85:TT85-A12-4, tt85:TT85-A11-3.b, tt85:TT85-A11-3, tt85:TT85-A12-6, tt85:TT85-A12-1, tt85:TT85-A12-2, tt85:TT85-A6-5..."
146,LQ147,legal_evidence,True,True,1.0,1.00,True,1.0,[],88.032708,"[tt85:TT85-A12-6, tt85:TT85-A5-1, tt85:TT85-A11-3, tt85:TT85-A11-3.b, tt85:TT85-A12-3, tt85:TT85-A12-1, tt85:TT85-A12-2, tt85:TT85-A12-4..."
147,LQ148,legal_evidence,True,True,1.0,1.00,True,1.0,[],88.869083,"[tt85:TT85-A12-7, tt85:TT85-A11-3.b, tt85:TT85-A11-3, tt85:TT85-A12-3, tt85:TT85-A12-6, tt85:TT85-A17-2, tt85:TT85-A11-3.c, tt85:TT85-A3..."
148,LQ149,legal_evidence,True,True,1.0,1.00,True,1.0,[],123.002500,"[tt85:TT85-A15-1, tt85:TT85-A10-2, tt85:TT85-A12-3, tt85:TT85-A1-1, tt85:TT85-A12-5, tt85:TT85-A15-2, tt85:TT85-A16-2, tt85:TT85-A1-2, t..."


## QA holdout — đánh giá chính sách đã khóa

100 câu trong `data/verified/qa/holdout` chỉ được đọc **sau** khi chọn cấu hình bằng QA train và đánh giá test. Chấm độ đầy đủ của các primary chunk trong Top-5/Top-10 và gói căn cứ, kèm kết quả từng câu. Không dùng holdout để đổi trọng số.


In [22]:
import hashlib

holdout_path = ROOT / "data/verified/qa/holdout/LEGAL_RETRIEVAL_HOLDOUT.jsonl"
qa_holdout_hash = hashlib.sha256(holdout_path.read_bytes()).hexdigest()
qa_holdout = discover_qa(bundle, split="holdout")
assert len(qa_holdout) == 100
assert not ({c["case_family_id"] for c in qa_holdout} &
            {c["case_family_id"] for c in qa_train + qa_test_current})
assert digest(read(ROOT / "config/selected_retrieval.json")) == policy_hash
holdout_results = []
for index, case in enumerate(qa_holdout, 1):
    with capture_output():
        result = runtime.qa_search(case["query"])
    direct = qa_direct_coverage(case, result["matches"])
    package = qa_coverage(case, result["matches"], result["package"])
    holdout_results.append({"id": case["id"], "category": case["category"],
        "case_type": case.get("case_type"),
        "direct_all@5": direct["all_groups@5"],
        "direct_all@10": direct["all_groups@10"],
        "group_recall@10": direct["group_recall@10"],
        "mrr@10": direct["mrr@10"],
        "package_all": package["package_all_groups"],
        "package_group_recall": package["package_group_recall"],
        "missing_package": package["missing_package_groups"],
        "top10_ids": direct["returned_top10"],
        "latency_ms": result["elapsed_ms"]})
    if index % 25 == 0:
        print("QA HOLDOUT", index, "/", len(qa_holdout), flush=True)
assert digest(read(ROOT / "config/selected_retrieval.json")) == policy_hash
holdout_frame = pd.DataFrame(holdout_results)
print("QA holdout file SHA256:", qa_holdout_hash)
print("Frozen policy:", policy_hash, "snapshot:", runtime.snapshot)
print("Overall", {"questions": len(holdout_frame),
    "direct_all@5": holdout_frame["direct_all@5"].mean(),
    "direct_all@10": holdout_frame["direct_all@10"].mean(),
    "group_recall@10": holdout_frame["group_recall@10"].mean(),
    "mrr@10": holdout_frame["mrr@10"].mean(),
    "package_all": holdout_frame["package_all"].mean(),
    "latency_p50_ms": holdout_frame["latency_ms"].median(),
    "latency_p95_ms": holdout_frame["latency_ms"].quantile(.95)})
display(holdout_frame.groupby("case_type")[["direct_all@10", "group_recall@10", "package_all"]].agg(["mean", "count"]))
display(holdout_frame)
conn.close()


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


QA HOLDOUT 25 / 100


QA HOLDOUT 50 / 100


QA HOLDOUT 75 / 100


QA HOLDOUT 100 / 100


QA holdout file SHA256: febca90b195426730935b4057cbf38de9e176ea54a6af8a8d6d24dbbb9bf8458
Frozen policy: 874e876c632e9356b2e93fbcba1c833c3974e13c3beb105d4c6341a4edd70d14 snapshot: e2354dac4a5bdc2c717cac99
Overall {'questions': 100, 'direct_all@5': np.float64(0.95), 'direct_all@10': np.float64(0.97), 'group_recall@10': np.float64(0.97), 'mrr@10': np.float64(0.8411666666666667), 'package_all': np.float64(0.97), 'latency_p50_ms': np.float64(992.0134999993024), 'latency_p95_ms': np.float64(1476.703752099456)}


direct_all@10       group_recall@10       package_all      
                        mean count            mean count        mean count
case_type                                                                 
gir                 1.000000    12        1.000000    12    1.000000    12
note_exclusion      1.000000     6        1.000000     6    1.000000     6
note_rule           1.000000    24        1.000000    24    1.000000    24
tariff_direct       0.933333    45        0.933333    45    0.933333    45
tt85                1.000000    13        1.000000    13    1.000000    13

,id,category,case_type,direct_all@5,direct_all@10,group_recall@10,mrr@10,package_all,package_group_recall,missing_package,top10_ids,latency_ms
0,H001,legal_evidence,tariff_direct,True,True,1.0,1.000000,True,1.0,[],"[tt31:HS4:7103, tt31:HS4:7104, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7101, tt31:HS4:7102, tt31:CH:71, tt31:XIV_CH71_NOTE_03, tt31:XIV_CH71_...",5100.362042
1,H002,legal_evidence,tariff_direct,True,True,1.0,1.000000,True,1.0,[],"[tt31:HS4:7104, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7103, tt31:HS4:7101, tt31:XIV_CH71_NOTE_11, tt31:HS4:7116, tt31:XIV_CH71_NOTE_03, tt3...",1560.050000
2,H003,legal_evidence,tariff_direct,True,True,1.0,1.000000,True,1.0,[],"[tt31:HS4:7108, tt31:HS4:7106, tt31:HS4:7110, tt31:HS4:7111, tt31:HS6:711011, tt31:HS6:711021, tt31:HS8:71090000, tt31:HS6:711041, tt31:...",1284.820458
3,H004,legal_evidence,tariff_direct,True,True,1.0,1.000000,True,1.0,[],"[tt31:HS4:7116, tt31:XIV_CH71_NOTE_09, tt31:SXVI-N-SXVI_NOTE_1F, tt31:XIV_CH71_NOTE_01, tt31:CH:71, tt31:XIV_CH71_NOTE_11, tt31:HS4:7101...",1317.480042
4,H005,legal_evidence,tariff_direct,False,True,1.0,0.166667,True,1.0,[],"[tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_02, tt31:CH:71, tt31:HS4:7113, tt31:XIV_CH71_NOTE_09, tt31:HS4:7117, tt31:SXVI-N-SXVI_NOTE_1F...",1349.345541
...,...,...,...,...,...,...,...,...,...,...,...,...
95,H096,legal_evidence,tt85,True,True,1.0,1.000000,True,1.0,[],"[tt85:TT85-A11-3.d, tt85:TT85-A11-3.b, tt85:TT85-A11-3, tt85:TT85-A11-3.c, tt85:TT85-A11-4, tt85:TT85-A16-1, tt85:TT85-A11-5, tt85:TT85-...",1357.578250
96,H097,legal_evidence,tt85,True,True,1.0,1.000000,True,1.0,[],"[tt85:TT85-A12-4, tt85:TT85-A12-3, tt85:TT85-A11-3.b, tt85:TT85-A11-3, tt85:TT85-A16-1, tt85:TT85-A12-6, tt85:TT85-A12-1, tt85:TT85-A12-...",1415.050000
97,H098,legal_evidence,tt85,True,True,1.0,1.000000,True,1.0,[],"[tt85:TT85-A13-3, tt85:TT85-A13-1, tt85:TT85-A13-2, tt85:TT85-A13-6, tt85:TT85-A13-4, tt85:TT85-A13-5, tt85:TT85-A14-3, tt85:TT85-A5-1, ...",1256.564875
98,H099,legal_evidence,tt85,True,True,1.0,1.000000,True,1.0,[],"[tt85:TT85-A15-2, tt85:TT85-A5-2, tt85:TT85-A16-1, tt85:TT85-A15-1, tt85:TT85-A6-5, tt85:TT85-A14-2, tt85:TT85-A1-2, tt85:TT85-A1-1, tt8...",1176.796542
